In [20]:
import glob
import numpy as np
import pandas as pd
import torch
from models.functions import load_best_ae_and_stats, autoencoder_inpaint
from sklearn.preprocessing import StandardScaler

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from scipy import stats

In [21]:
# Simple VAE for (2, 10) kernels -> (4,) latent space
class SimpleVAE(nn.Module):
    def __init__(self, input_dim=20, latent_dim=6):
        super().__init__()
        # Encoder
        self.fc1 = nn.Linear(input_dim, 16)
        self.fc_mu = nn.Linear(16, latent_dim)
        self.fc_logvar = nn.Linear(16, latent_dim)
        
        # Decoder
        self.fc3 = nn.Linear(latent_dim, 16)
        self.fc4 = nn.Linear(16, input_dim)
    
    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        h = F.relu(self.fc3(z))
        return self.fc4(h)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [22]:
# Load metadata
df = pd.read_csv("data/hea.csv", index_col=0)

# Load reconstructed model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, model_stats = load_best_ae_and_stats(
    "results/autoencoder/grid_search/best_inpainting_model.pth", 
    device
)

# Load 15% dataset
DATABASE_PATH = "data/zero_quantiles/0.15_data/*.dat"
all_signals = {}
fhr_gain, fhr_baseline = 100.0, 0.0

for dat_file in glob.glob(DATABASE_PATH):
    rec_id = int(dat_file.split("/")[-1].split(".")[0])
    raw_data = np.fromfile(dat_file, dtype=np.int16)
    fhr = (raw_data[0::2] - fhr_baseline) / fhr_gain
    uc = raw_data[1::2]
    
    # Reconstruct FHR signal
    fhr_reconstructed = autoencoder_inpaint(fhr, uc, model, model_stats, device)
    
    # Cut UC to match reconstructed FHR length
    uc = uc[:len(fhr_reconstructed)]
    
    all_signals[rec_id] = {
        "fhr": fhr_reconstructed,
        "uc": uc,
        "label": df[df["metadata.id"] == rec_id]["fetal_state"].iloc[0]
    }

print(f"Loaded {len(all_signals)} signals")

Loading model from results/autoencoder/grid_search/best_inpainting_model.pth...
Model loaded successfully.
Stats: FHR μ=137.47, σ=17.29 | UC μ=2195.35, σ=2203.97
Loaded 228 signals


In [23]:
# Create sliding window kernels (10 timeframes x 2 channels)
KERNEL_SIZE = 10
STRIDE = KERNEL_SIZE//2  # 50% overlap

def create_kernels(signals_dict):
    kernels = []
    labels = []
    rec_ids = []
    
    for rec_id, data in signals_dict.items():
        fhr = data["fhr"]
        uc = data["uc"]
        label = data["label"]
        
        # Ensure same length
        min_len = min(len(fhr), len(uc))
        fhr = fhr[:min_len]
        uc = uc[:min_len]
        
        # Slide window across signal
        for i in range(0, min_len - KERNEL_SIZE + 1, STRIDE):
            kernel = np.stack([
                fhr[i:i+KERNEL_SIZE],
                uc[i:i+KERNEL_SIZE]
            ], axis=0)  # Shape: (2, 10)
            
            kernels.append(kernel)
            labels.append(label)
            rec_ids.append(rec_id)
    
    return np.array(kernels), np.array(labels), np.array(rec_ids)

kernels, labels, rec_ids = create_kernels(all_signals)
print(f"Created {len(kernels)} kernels of shape {kernels[0].shape}")

Created 794067 kernels of shape (2, 10)


In [24]:
# Filter healthy kernels only
healthy_mask = labels == "normal"
healthy_kernels = kernels[healthy_mask]

# Flatten kernels from (2, 10) -> (20,)
healthy_kernels_flat = healthy_kernels.reshape(len(healthy_kernels), -1)

scaler = StandardScaler()
healthy_kernels_flat = scaler.fit_transform(healthy_kernels_flat)

# Train/test split
X_train, X_test = train_test_split(healthy_kernels_flat, test_size=0.2, random_state=42)

# Convert to tensors
train_dataset = TensorDataset(torch.FloatTensor(X_train))
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize model
vae = SimpleVAE(input_dim=20, latent_dim=4).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

# VAE Loss
def vae_loss(recon_x, x, mu, logvar, beta=0.1):
    recon_loss = F.mse_loss(recon_x, x, reduction='mean')  # Changed to 'mean'
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kld

# Training loop
EPOCHS = 5
vae.train()
for epoch in range(EPOCHS):
    train_loss = 0
    for batch in train_loader:
        x = batch[0].to(device)
        optimizer.zero_grad()
        recon, mu, logvar = vae(x)
        loss = vae_loss(recon, x, mu, logvar)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    if (epoch + 1) % 1 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {train_loss/len(train_loader):.4f}")

print("VAE training complete!")

Epoch 1/5, Loss: 0.1542
Epoch 2/5, Loss: 0.1274
Epoch 3/5, Loss: 0.1272
Epoch 4/5, Loss: 0.1270
Epoch 5/5, Loss: 0.1270
VAE training complete!


In [30]:
# Calculate reconstruction error for all kernels
vae.eval()
all_kernels_flat = kernels.reshape(len(kernels), -1)

# FIX: Scale the full dataset using the scaler fitted on healthy data
all_kernels_scaled = scaler.transform(all_kernels_flat)
all_tensors = torch.FloatTensor(all_kernels_scaled).to(device)

with torch.no_grad():
    recon, _, _ = vae(all_tensors)
    reconstruction_errors = F.mse_loss(recon, all_tensors, reduction='none').mean(dim=1).cpu().numpy()

# Calculate baseline (healthy) reconstruction error distribution
healthy_errors = reconstruction_errors[healthy_mask]
healthy_mean = healthy_errors.mean()
healthy_std = healthy_errors.std()

print(f"Healthy reconstruction error: {healthy_mean:.4f} ± {healthy_std:.4f}")

# Calculate p-values (assuming normal distribution)
# p-value: probability that error comes from healthy distribution
z_scores = (reconstruction_errors - healthy_mean) / healthy_std
p_values = 1 - stats.norm.cdf(z_scores)  # One-tailed test

# Aggregate results by recording
results = pd.DataFrame({
    'rec_id': rec_ids,
    'recon_error': reconstruction_errors,
    'p_value': p_values,
    'label': labels
})


Healthy reconstruction error: 0.0411 ± 0.0676


In [ ]:
thresholds = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
percentages = [0.01, 0.05, 0.1, 0.2]

best_acc = 0

for p_value_threshold in thresholds:
    for percentage_threshold in percentages:

        # Summary statistics per recording
        # Mark each segment as anomalous based on the p-value threshold
        results['is_anomalous'] = (results['p_value'] < p_value_threshold).astype(int)

        # Group by recording and calculate the percentage of anomalous segments
        summary = results.groupby(['rec_id', 'label'])['is_anomalous'].mean().reset_index()
        summary.rename(columns={'is_anomalous': 'anomalous_segment_percentage'}, inplace=True)

        # Classification based on the percentage of anomalous segments
        summary['suspicious_regions'] = (summary['anomalous_segment_percentage'] > percentage_threshold).astype(int)

        # Evaluate classification performance
        y_true = (summary['label'] == 'abnormal').astype(int)
        y_pred = summary['suspicious_regions']
        acc = (y_true==y_pred).mean()

        if acc > best_acc:
            print("**New Best")
            best_acc = acc
            best_sum = summary

        print(f"P value threshold: {p_value_threshold}, Percentage threshold: {percentage_threshold}, Accuracy: {acc}")

**New Best
P value threshold: 0.01, Percentage threshold: 0.01, Accuracy: 0.42543859649122806
**New Best
P value threshold: 0.01, Percentage threshold: 0.05, Accuracy: 0.6271929824561403
**New Best
P value threshold: 0.01, Percentage threshold: 0.1, Accuracy: 0.6447368421052632
**New Best
P value threshold: 0.01, Percentage threshold: 0.2, Accuracy: 0.6491228070175439
P value threshold: 0.05, Percentage threshold: 0.01, Accuracy: 0.39035087719298245
P value threshold: 0.05, Percentage threshold: 0.05, Accuracy: 0.6008771929824561
P value threshold: 0.05, Percentage threshold: 0.1, Accuracy: 0.6359649122807017
P value threshold: 0.05, Percentage threshold: 0.2, Accuracy: 0.6491228070175439
P value threshold: 0.1, Percentage threshold: 0.01, Accuracy: 0.35526315789473684
P value threshold: 0.1, Percentage threshold: 0.05, Accuracy: 0.5701754385964912
P value threshold: 0.1, Percentage threshold: 0.1, Accuracy: 0.618421052631579
P value threshold: 0.1, Percentage threshold: 0.2, Accuracy:

In [38]:
print('Summary by recording (showing percentage of anomalous segments):')
print(summary.head(10))

Summary by recording (showing percentage of anomalous segments):
   rec_id     label  anomalous_segment_percentage  suspicious_regions
0    1004    normal                      0.375707                   1
1    1008  abnormal                      0.271212                   1
2    1010    normal                      0.237273                   1
3    1011  abnormal                      0.339532                   1
4    1019  abnormal                      0.325097                   1
5    1020  abnormal                      0.320929                   1
6    1021    normal                      0.210182                   1
7    1022    normal                      0.262568                   1
8    1023    normal                      0.295916                   1
9    1024  abnormal                      0.277974                   1


In [39]:
# Evaluate classification performance
y_true = (summary['label'] == 'abnormal').astype(int)
y_pred = summary['suspicious_regions']

print(classification_report(y_true, y_pred, target_names=['Normal', 'Abnormal']))

              precision    recall  f1-score   support

      Normal       0.70      0.05      0.09       148
    Abnormal       0.35      0.96      0.52        80

    accuracy                           0.37       228
   macro avg       0.53      0.50      0.30       228
weighted avg       0.58      0.37      0.24       228

